# ACE-Step 1.5 Music Generator (Kaggle)

Reads the `music` block from `storyboard.json` and generates 3 instrumental candidates (`candidate_1.wav`, `candidate_2.wav`, `candidate_3.wav`) in `outputs/music/` with different seeds.

- Tempo (`bpm`) and key (`keyscale`) are passed in `GenerationParams` settings fields, not inside `caption`.
- Logs per-candidate generation time to `outputs/music/acestep_log.json`.
- After listening, copy your pick to `outputs/music/chosen.wav` for `pipeline/assemble.py`.

In [ ]:
!git clone -q https://github.com/ace-step/ACE-Step-1.5.git /kaggle/working/ACE-Step-1.5
%pip install -q -e /kaggle/working/ACE-Step-1.5

In [ ]:
import json
from pathlib import Path
import shutil
import sys
import time

sys.path.insert(0, "/kaggle/working/ACE-Step-1.5")

PROJECT_DIR = Path("/kaggle/working/short") if Path("/kaggle/working/short").exists() else Path.cwd().parent
STORYBOARD_PATH = PROJECT_DIR / "storyboard.json"
MUSIC_DIR = PROJECT_DIR / "outputs" / "music"
LOG_PATH = MUSIC_DIR / "acestep_log.json"
ACESTEP_ROOT = Path("/kaggle/working/ACE-Step-1.5")

MUSIC_DIR.mkdir(parents=True, exist_ok=True)
storyboard = json.loads(STORYBOARD_PATH.read_text(encoding="utf-8"))
music = storyboard["music"]

caption = music["caption"]
bpm = int(music["bpm"])
keyscale = music["key"]
duration = float(music["duration_seconds"])
instrumental = bool(music.get("instrumental", True))
base_seed = int(music.get("seed", 42))
seeds = [base_seed, base_seed + 1, base_seed + 2]

print(f"Caption: {caption}")
print(f"Settings -> bpm={bpm}, keyscale='{keyscale}', duration={duration}s, instrumental={instrumental}, seeds={seeds}")

In [ ]:
from acestep.handler import AceStepHandler
from acestep.llm_inference import LLMHandler
from acestep.inference import GenerationParams, GenerationConfig, generate_music

dit_handler = AceStepHandler()
dit_handler.initialize_service(
    project_root=str(ACESTEP_ROOT),
    config_path="acestep-v15-turbo",
    device="cuda",
)

llm_handler = LLMHandler()
llm_handler.initialize(
    checkpoint_dir=str(ACESTEP_ROOT / "checkpoints"),
    lm_model_path="acestep-5Hz-lm-0.6B",
    backend="pt",
    device="cuda",
)

In [ ]:
logs = []
for idx, seed in enumerate(seeds, start=1):
    candidate_path = MUSIC_DIR / f"candidate_{idx}.wav"
    if candidate_path.exists() and candidate_path.stat().st_size > 0:
        print(f"candidate_{idx}: skipped (already exists)")
        continue

    params = GenerationParams(
        task_type="text2music",
        caption=caption,
        lyrics="",
        instrumental=instrumental,
        bpm=bpm,
        keyscale=keyscale,
        duration=duration,
        seed=seed,
    )
    config = GenerationConfig(
        batch_size=1,
        audio_format="wav",
        seeds=[seed],
    )

    t0 = time.perf_counter()
    result = generate_music(
        dit_handler=dit_handler,
        llm_handler=llm_handler,
        params=params,
        config=config,
        save_dir=str(MUSIC_DIR),
    )
    elapsed = round(time.perf_counter() - t0, 3)
    if not result.success or not result.audios:
        raise RuntimeError(f"candidate_{idx} failed: {result.error}")

    src_path = Path(result.audios[0]["path"])
    shutil.move(str(src_path), str(candidate_path))
    entry = {
        "candidate": f"candidate_{idx}",
        "file": candidate_path.name,
        "seed": seed,
        "bpm": bpm,
        "keyscale": keyscale,
        "duration": duration,
        "latency_seconds": elapsed,
    }
    logs.append(entry)
    print(f"candidate_{idx} (seed={seed}): saved {candidate_path.name} in {elapsed}s")

LOG_PATH.write_text(json.dumps(logs, indent=2), encoding="utf-8")
print(f"Saved timing log to {LOG_PATH}")